# Nodal SDS shot detection — position-coded SDS, named time-window batch version

This notebook searches only the named active-source time ranges using the derived position-coded SDS archive.

Main design:

- Uses `nodal_sds_position_codes`, where station codes are 5-character position-in-cm codes, e.g. `09605` = 96.05 m.
- Outer loop = one named survey/source window at a time.
- Inner loop = short waveform chunks, default 60 s, with 10 s overlap.
- Each label gets its own output tree.
- Network/location are inferred from the label prefix, e.g. `T1_N2_Refraction1m` → network `T1`, location `N2`.
- Geometry is derived from station codes, so the notebook no longer depends on fragile station/serial columns in the Excel geometry sheets.
- The supplied field/laptop time windows are buffered by 60 s before and 180 s after.
- Whole-deployment streams are never loaded.

Notes:

- `DP*` channels are the 500 Hz nodal components. Use `GP*` if you want the 1000 Hz channels.
- Moving trigger/source nodes such as `12806` should not be treated as fixed-position receivers. They are excluded if encountered.


## 1. Imports

In [1]:
from pathlib import Path
import gc
import traceback

import numpy as np
import pandas as pd
import obspy
from obspy import UTCDateTime

try:
    import psutil
except Exception:
    psutil = None

from nodal_shotgather import (
    DetectionConfig,
    PickingConfig,
    read_deployment_from_sds,
    load_station_geometry_table,
    geometry_dataframe_to_dict,
    attach_station_geometry,
    save_station_geometry_csv,
    preprocess_for_detection,
    detect_network_events,
    preprocess_for_picking,
    run_station_picking,
    trim_events_from_consensus_picks,
    save_event_mseed_files,
    save_event_qc_figures,
)


## 2. Paths and named time windows

In [2]:
# -----------------------------------------------------------------------------
# Input data
# -----------------------------------------------------------------------------
sds_root = Path("/Volumes/tachyon/LBSSP_DATA/nodal_sds_position_codes")

# Geometry workbook generated earlier.
box_folder_root = Path("/Users/glennthompson/Library/CloudStorage/Box-Box/thompsong/2026KarstGeophysicsDEP")
seismic_analysis_dir = box_folder_root / "04_FieldData"
metadata_workbook = seismic_analysis_dir / "glenn_smartsolo_nodal_metadata_with_estimated_coords.xlsx"

# -----------------------------------------------------------------------------
# Output root. Each time-window label gets its own subfolder below this.
# -----------------------------------------------------------------------------
base_outdir = Path("/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows")
base_outdir.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Time windows to search. These should be UTC SmartSolo/SDS times.
# -----------------------------------------------------------------------------
timewindows = { 
    "T1_N1_Streamer": (UTCDateTime("2026-05-16T17:13:00"), UTCDateTime("2026-05-16T21:48:00")),
    "T1_N2_Nodal1": (UTCDateTime("2026-05-17T16:00:00"), UTCDateTime("2026-05-17T19:00:00")),
    "T1_N2_Refraction1m": (UTCDateTime("2026-05-18T16:03:54"), UTCDateTime("2026-05-18T18:38:33")),
    "T1_N2_Refraction2m": (UTCDateTime("2026-05-18T20:18:44"), UTCDateTime("2026-05-18T23:12:53")),
    "T1_N2_Nodal2": (UTCDateTime("2026-05-19T12:59:00"), UTCDateTime("2026-05-19T13:21:00")),
    "T1_N3_Nodal3": (UTCDateTime("2026-05-19T13:59:00"), UTCDateTime("2026-05-19T14:15:00")),
    "T3_N4_Refraction1am": (UTCDateTime("2026-05-19T16:02:00"), UTCDateTime("2026-05-19T18:25:00")),
}

# Search padding for unsynced laptop/file-stack timing.
SEARCH_PAD_BEFORE_S = 60
SEARCH_PAD_AFTER_S = 180

# Leave as None to run all windows. Or set e.g. ["T1_N2_Nodal2"] for testing.
RUN_LABELS = None

# Station/component selection.
station = "*"
channel = "DP*"      # DP* = 500 Hz components. Use GP* for 1000 Hz, or DPZ/GPZ for vertical only.


# Exclude moving source/trigger nodes from receiver geometry if they appear.
# 12806 / 012806 / 45012806 was used as a trigger node and moved shot-to-shot.
EXCLUDE_STATIONS = {"12806", "012806", "45012806"}

# In the position-coded SDS archive, station codes encode x position in cm.
POSITION_CODED_STATIONS = True
# Geometry sheet by network/location. Used only as a fallback if POSITION_CODED_STATIONS=False.
geometry_sheet_by_deployment = {
    ("T1", "N1"): "T1_Nodal_Geometry_Orig",
    ("T1", "N2"): "T1_Nodal_Geometry_DenseConfig",
    ("T1", "N3"): "T1_Nodal_Geometry_Orig",
    ("T3", "N4"): "T3_Nodal_Geometry",
}

print(f"Base output directory: {base_outdir}")
print(f"Number of named windows: {len(timewindows)}")


Base output directory: /Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows
Number of named windows: 7


## 3. Detection, picking, and chunking parameters

In [3]:
detect_cfg = DetectionConfig(
    freqmin=5.0,
    freqmax=200.0,
    sta_seconds=0.02,
    lta_seconds=0.30,
    threshold_on=4.0,
    threshold_off=1.5,
    min_channels=12,       # total channel detections, not necessarily stations
    min_snr=10.0,
    pretrigger_seconds=0.25,
    posttrigger_seconds=0.25,
    min_event_peak_amplitude=0.0,
)

pick_cfg = PickingConfig(
    freqmin=10.0,
    freqmax=150.0,
    pick_tolerance_s=0.02,
    min_votes=2,
    include_ar_s=False,
    event_pre_pick_s=0.05,
    event_length_s=0.75,
)

chunk_seconds = 60          # 60 s is safest. Increase later if comfortable.
overlap_seconds = 10        # read overlap around each chunk to avoid edge losses.
max_chunk_seconds = 3600    # hard safety limit.

duplicate_tolerance_s = 0.20

write_event_mseed = True
make_detection_quicklooks = False
make_event_qc_figures = False
max_quicklooks_per_window = 10

# Time-window padding because supplied times come from a laptop clock
# and Geode file times generally refer to the first shot in a stack.
SEARCH_PAD_BEFORE_S = 60
SEARCH_PAD_AFTER_S = 180

# Set True to regenerate existing per-chunk detection CSV files.
# Useful after changing padding, thresholds, or output columns.
reprocess_existing_chunks = False

assert chunk_seconds <= max_chunk_seconds, "chunk_seconds must be <= 3600 seconds"
assert overlap_seconds >= 0
assert overlap_seconds < chunk_seconds, "overlap_seconds should be shorter than chunk_seconds"


## 4. Utility functions

In [4]:
def memory_status(prefix=""):
    """Print a lightweight RAM status message if psutil is available."""
    if psutil is None:
        return
    vm = psutil.virtual_memory()
    print(f"{prefix}RAM used={vm.used/1e9:.1f} GB, available={vm.available/1e9:.1f} GB, percent={vm.percent:.1f}%")


def iter_time_chunks(start, end, chunk_s):
    """Yield non-overlapping core chunks [t0, t1). Overlap is added only for reading."""
    t0 = UTCDateTime(start)
    end = UTCDateTime(end)
    while t0 < end:
        t1 = min(t0 + float(chunk_s), end)
        yield t0, t1
        t0 = t1


def parse_label(label):
    """Infer network/location from labels such as T1_N2_Refraction1m."""
    parts = label.split("_")
    if len(parts) < 2:
        raise ValueError(f"Cannot infer network/location from label: {label}")
    network, location = parts[0], parts[1]
    return network, location


def make_output_dirs(label, network, location):
    """Create and return per-label output directories."""
    channel_tag = channel.replace("*", "all")
    outdir = base_outdir / label / f"{network}_{location}_{channel_tag}"
    dirs = {
        "outdir": outdir,
        "figs": outdir / "figures",
        "tables": outdir / "tables",
        "chunks": outdir / "tables" / "chunk_detections",
        "events_mseed": outdir / "events_mseed",
        "logs": outdir / "logs",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs


def _normalize_name(x):
    return str(x).strip().lower().replace(" ", "_").replace("-", "_")


def _find_first_existing_column(df, candidates):
    # exact match first
    for c in candidates:
        if c in df.columns:
            return c

    # normalized match second
    norm = {_normalize_name(c): c for c in df.columns}
    for c in candidates:
        key = _normalize_name(c)
        if key in norm:
            return norm[key]
    return None


def event_time_columns(df: pd.DataFrame):
    start_candidates = [
        "event_start", "start_time", "starttime",
        "on_time", "time_on", "trigger_on"
    ]
    end_candidates = [
        "event_end", "end_time", "endtime",
        "off_time", "time_off", "trigger_off"
    ]
    center_candidates = [
        "event_center", "center_time", "centertime",
        "peak_time", "time"
    ]

    start_col = _find_first_existing_column(df, start_candidates)
    end_col = _find_first_existing_column(df, end_candidates)
    center_col = _find_first_existing_column(df, center_candidates)

    if start_col is None:
        raise ValueError(
            f"Cannot infer event start-time column from columns: {list(df.columns)}"
        )

    return start_col, end_col, center_col


def infer_pick_time_column(df: pd.DataFrame):
    """Find the best pick-time column in station consensus picks.

    The current run_station_picking() output uses consensus_time.
    Older/alternate tables may use pick_time or arrival_time.
    """
    candidates = [
        "consensus_time",
        "pick_time", "consensus_pick_time", "station_pick_time",
        "arrival_time", "time", "event_on_time", "on_time",
    ]
    col = _find_first_existing_column(df, candidates)
    if col is None:
        raise ValueError(
            f"Cannot infer pick-time column from station_picks_df columns: {list(df.columns)}"
        )
    return col


def to_utcdatetime(x):
    if isinstance(x, UTCDateTime):
        return x
    if pd.isna(x):
        return None
    return UTCDateTime(str(x))


def add_event_timing_helpers(df):
    """Add numeric start/end/center seconds for sorting and de-duplication."""
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    start_col, end_col, center_col = event_time_columns(df)
    starts = [to_utcdatetime(v) for v in df[start_col]]
    ends = [to_utcdatetime(v) for v in df[end_col]] if end_col is not None else starts
    centers = []
    for s, e in zip(starts, ends):
        if s is None:
            centers.append(np.nan)
        elif e is None:
            centers.append(float(s.timestamp))
        else:
            centers.append(0.5 * (float(s.timestamp) + float(e.timestamp)))
    df["_event_start_ts"] = [float(s.timestamp) if s is not None else np.nan for s in starts]
    df["_event_end_ts"] = [float(e.timestamp) if e is not None else np.nan for e in ends]
    df["_event_center_ts"] = centers
    return df


def add_padding_status(df, nominal_start, nominal_end):
    """Mark whether each event falls before, inside, or after the nominal field-note window."""
    if df is None or len(df) == 0:
        return df
    df = add_event_timing_helpers(df)
    ns = float(UTCDateTime(nominal_start).timestamp)
    ne = float(UTCDateTime(nominal_end).timestamp)
    t = df["_event_center_ts"].fillna(df["_event_start_ts"])
    status = np.where(t < ns, "before_nominal", np.where(t > ne, "after_nominal", "inside_nominal"))
    df = df.copy()
    df["padding_status"] = status
    df["nominal_starttime"] = str(UTCDateTime(nominal_start))
    df["nominal_endtime"] = str(UTCDateTime(nominal_end))
    return df


def add_detection_score(df):
    """Build a best-effort detection score so duplicate overlap detections can be ranked."""
    if df is None or len(df) == 0:
        return df
    df = df.copy()
    score = np.zeros(len(df), dtype=float)
    for col, weight in [
        ("n_seed_ids", 25.0), ("cluster_size", 20.0),
        ("n_channels", 10.0), ("num_channels", 10.0), ("n_triggered", 10.0),
        ("n_stations", 20.0), ("num_stations", 20.0),
        ("coincidence_sum", 5.0),
        ("snr_rms", 1.0), ("snr", 1.0), ("max_snr", 1.0), ("median_snr", 1.0),
        ("signal_rms", 0.1),
        ("peak_amplitude", 0.01), ("max_amplitude", 0.01), ("amplitude", 0.01),
    ]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy()
            score += weight * vals
    df["_detection_score"] = score
    return df


def filter_events_to_core_chunk(df, core_start, core_end):
    """Keep events whose center/start falls within the non-overlap core chunk."""
    if df is None or len(df) == 0:
        return df
    df = add_event_timing_helpers(df)
    t0 = float(core_start.timestamp)
    t1 = float(core_end.timestamp)
    key = df["_event_center_ts"].fillna(df["_event_start_ts"])
    return df[(key >= t0) & (key < t1)].copy()


def merge_duplicate_events(df, duplicate_tolerance_s=0.20):
    """Merge duplicate detections caused by overlap or adjacent components."""
    if df is None or len(df) == 0:
        return pd.DataFrame()
    df = add_event_timing_helpers(df)
    df = add_detection_score(df)
    df = df.sort_values("_event_center_ts").reset_index(drop=True)

    groups = []
    current = []
    last_center = None
    for idx, row in df.iterrows():
        center = row["_event_center_ts"]
        if last_center is None or abs(center - last_center) <= duplicate_tolerance_s:
            current.append(idx)
        else:
            groups.append(current)
            current = [idx]
        last_center = center
    if current:
        groups.append(current)

    keep_indices = []
    for g in groups:
        sub = df.loc[g]
        keep_indices.append(sub["_detection_score"].idxmax())

    merged = df.loc[keep_indices].sort_values("_event_center_ts").reset_index(drop=True)
    merged["event_id"] = [f"E{i:05d}" for i in range(1, len(merged) + 1)]
    return merged


def _find_header_row_for_geometry(path, sheet_name, max_scan_rows=10):
    """Find the row containing geometry headers such as adopted_position_m or node_position_m."""
    preview = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=max_scan_rows)
    wanted = {
        "adopted_position_m", "node_position_m", "position_m",
        "station", "normalized_serial_number", "node_from_north"
    }
    for i in range(len(preview)):
        vals = {_normalize_name(v) for v in preview.iloc[i].dropna().tolist()}
        if len(vals & wanted) >= 2 or ("node_from_north" in vals and ("adopted_position_m" in vals or "node_position_m" in vals)):
            return i
    return 0


def _list_sds_stations_for_deployment(sds_root, network, location):
    """List station codes that appear in the SDS archive for network/location."""
    stations_found = set()
    sds_root = Path(sds_root)

    for year_dir in sorted(sds_root.glob("[0-9][0-9][0-9][0-9]")):
        net_dir = year_dir / network
        if not net_dir.exists():
            continue

        for sta_dir in sorted([p for p in net_dir.iterdir() if p.is_dir()]):
            found_location = False
            for chan_dir in sta_dir.iterdir():
                if not chan_dir.is_dir():
                    continue
                # Look at a few files only. The location code appears in filenames as .N1. etc.
                for f in chan_dir.iterdir():
                    if f.is_file() and f".{location}." in f.name:
                        found_location = True
                        break
                if found_location:
                    break
            if found_location:
                stations_found.add(sta_dir.name)

    return sorted(stations_found)


def _load_geometry_sheet_flexible(path, sheet_name, network, location):
    """Load geometry from messy workbook sheets and standardize columns for nodal_shotgather.

    The T1 sheets have a title row before the real header. They also may not have an SDS
    station-code column. In that case, assign station codes from the SDS archive in geometry order.
    """
    header_row = _find_header_row_for_geometry(path, sheet_name)
    df = pd.read_excel(path, sheet_name=sheet_name, header=header_row)
    df = df.dropna(how="all").copy()

    # Drop repeated header rows if they occur
    df.columns = [str(c).strip() for c in df.columns]
    norm_to_actual = {_normalize_name(c): c for c in df.columns}

    station_col = _find_first_existing_column(df, [
        "station", "seed_station", "seed_id", "seedid", "sds_station", "station_code",
        "smartsolo_station", "node_station"
    ])
    serial_col = _find_first_existing_column(df, [
        "normalized_serial_number", "raw_serial_number", "serial_number", "serial"
    ])
    node_col = _find_first_existing_column(df, [
        "node_from_north", "node", "node_id"
    ])
    x_col = _find_first_existing_column(df, [
        "adopted_position_m", "node_position_m", "position_m", "x_m",
        "line_x_m", "distance_m", "offset_m", "station_position_m"
    ])
    lat_col = _find_first_existing_column(df, ["latitude", "lat"])
    lon_col = _find_first_existing_column(df, ["longitude", "lon", "long"])

    if x_col is None:
        raise ValueError(
            f"Could not identify position/x column in sheet {sheet_name}. "
            f"Columns are: {list(df.columns)}"
        )

    out = pd.DataFrame()
    out["x_m"] = pd.to_numeric(df[x_col], errors="coerce")
    out = out.dropna(subset=["x_m"]).copy()

    # Preserve row order by along-line coordinate.
    # For T1 sheets, node_from_north and x_m are both usable; x_m is what we need.
    if node_col is not None:
        out["node_from_north"] = df.loc[out.index, node_col].values

    if serial_col is not None:
        out["serial"] = df.loc[out.index, serial_col].astype(str).str.strip().values

    if lat_col is not None:
        out["latitude"] = pd.to_numeric(df.loc[out.index, lat_col], errors="coerce").values
    if lon_col is not None:
        out["longitude"] = pd.to_numeric(df.loc[out.index, lon_col], errors="coerce").values

    if station_col is not None:
        out["station"] = df.loc[out.index, station_col].astype(str).str.strip().values
    else:
        sds_stations = _list_sds_stations_for_deployment(sds_root, network, location)
        out = out.sort_values("x_m").reset_index(drop=True)

        if len(sds_stations) == len(out):
            out["station"] = sds_stations
            print(f"Assigned {len(sds_stations)} SDS station codes to geometry rows in position order.")
        elif len(sds_stations) > 0:
            n = min(len(sds_stations), len(out))
            print(
                f"WARNING: geometry rows ({len(out)}) and SDS stations ({len(sds_stations)}) differ for "
                f"{network}.{location}. Assigning first {n} station codes in sorted order."
            )
            out = out.iloc[:n].copy()
            out["station"] = sds_stations[:n]
        elif serial_col is not None:
            print("WARNING: Could not find SDS station codes; falling back to serial numbers as station IDs.")
            out["station"] = out["serial"].astype(str)
        elif node_col is not None:
            print("WARNING: Could not find SDS station codes; falling back to node numbers as station IDs.")
            out["station"] = [f"NODE{int(v):03d}" if pd.notna(v) else f"ROW{i:03d}" for i, v in enumerate(out["node_from_north"])]
        else:
            raise ValueError("Could not identify station, serial, node, or SDS station list.")

    # Remove bad rows and duplicates
    out["station"] = out["station"].astype(str).str.strip()
    out = out[out["station"].ne("") & out["station"].ne("nan")].copy()
    out = out.drop_duplicates(subset=["station"], keep="first")
    out = out.sort_values("x_m").reset_index(drop=True)

    return out



def _station_is_excluded(sta):
    variants = {str(sta), str(sta).lstrip("0"), str(sta).zfill(5)}
    return bool(variants & set(EXCLUDE_STATIONS))


def station_code_to_position_m(station_code):
    """Convert 5-character position-coded station ID to metres.

    Example: 09605 -> 96.05 m.
    """
    s = str(station_code).strip()
    if not s.isdigit():
        raise ValueError(f"Station code is not numeric: {station_code}")
    if len(s) > 5:
        # Allow accidental longer codes only if they are obviously old serials;
        # position-coded receiver stations should normally be exactly 5 digits.
        raise ValueError(f"Station code is longer than 5 digits and is not position-coded: {station_code}")
    return int(s) / 100.0


def discover_position_coded_stations_from_sds(sds_root, network, location, channel_pattern="DP*"):
    """Return sorted position-coded stations present for network/location/channel.

    This inspects SDS filenames rather than loading waveform data.
    Expected filename: NET.STA.LOC.CHA.D.YEAR.JDAY
    """
    import fnmatch

    root = Path(sds_root)
    stations = set()
    for year_dir in sorted(root.glob("[0-9][0-9][0-9][0-9]")):
        net_dir = year_dir / network
        if not net_dir.exists():
            continue
        for sta_dir in net_dir.iterdir():
            if not sta_dir.is_dir():
                continue
            sta = sta_dir.name
            if _station_is_excluded(sta):
                continue
            if not sta.isdigit() or len(sta) != 5:
                # Skip non-position-coded stations if any remain.
                continue
            for chan_dir in sta_dir.iterdir():
                if not chan_dir.is_dir():
                    continue
                chan = chan_dir.name.split(".")[0]
                if not fnmatch.fnmatch(chan, channel_pattern):
                    continue
                pattern = f"{network}.{sta}.{location}.{chan}.D.*.*"
                if any(chan_dir.glob(pattern)):
                    stations.add(sta)
                    break
    return sorted(stations, key=lambda s: int(s))


def load_position_coded_geometry_for_deployment(network, location, dirs):
    """Build geometry directly from position-coded SDS station names.

    Returns the same tuple shape as the earlier workbook-based loader.
    """
    station_names = discover_position_coded_stations_from_sds(sds_root, network, location, channel)
    if not station_names:
        raise ValueError(
            f"No position-coded stations found in {sds_root} for {network}.{location}.{channel}. "
            "Check sds_root, location code, channel pattern, or whether the conversion has been run."
        )

    geom_df = pd.DataFrame({
        "station": station_names,
        "x_m": [station_code_to_position_m(s) for s in station_names],
        "y_m": np.zeros(len(station_names)),
        "elevation_m": np.nan,
        "latitude": np.nan,
        "longitude": np.nan,
    })

    # Try the project utility first. If its expected columns differ, fall back to
    # a simple dict that attach_station_geometry should be able to use.
    try:
        geometry = geometry_dataframe_to_dict(geom_df)
    except Exception:
        geometry = {
            r["station"]: {
                "x_m": float(r["x_m"]),
                "y_m": float(r["y_m"]),
                "elevation_m": np.nan,
                "latitude": np.nan,
                "longitude": np.nan,
            }
            for _, r in geom_df.iterrows()
        }

    stations_csv = dirs["tables"] / f"{network}_{location}_position_coded_station_geometry.csv"
    try:
        save_station_geometry_csv(geometry, stations_csv)
    except Exception:
        geom_df.to_csv(stations_csv, index=False)

    geometry_source = "position_coded_sds_station_codes"
    return geometry, station_names, geometry_source, stations_csv


def load_geometry_for_deployment(network, location, dirs):
    """Load geometry for a deployment.

    Preferred mode: derive x position directly from 5-character station codes in
    the position-coded SDS archive. This is much more robust than matching
    serial-number station IDs through Excel sheets.
    """
    if POSITION_CODED_STATIONS:
        return load_position_coded_geometry_for_deployment(network, location, dirs)

    geometry_sheet = geometry_sheet_by_deployment[(network, location)]

    try:
        geom_df = load_station_geometry_table(
            metadata_workbook,
            sheet_name=geometry_sheet,
            station_col="station",
            serial_col="normalized_serial_number",
            x_col="adopted_position_m",
            latitude_col="latitude",
            longitude_col="longitude",
        )
    except Exception as exc:
        print(f"Strict geometry loader failed for {geometry_sheet}: {exc}")
        print("Using flexible geometry loader.")
        geom_df = _load_geometry_sheet_flexible(metadata_workbook, geometry_sheet, network, location)

    geometry = geometry_dataframe_to_dict(geom_df)
    station_names = sorted(geometry.keys())
    stations_csv = dirs["tables"] / f"{network}_{location}_station_geometry.csv"
    save_station_geometry_csv(geometry, stations_csv)
    return geometry, station_names, geometry_sheet, stations_csv


def filter_excluded_stations(st):
    """Remove moving trigger/source stations if they appear in a stream."""
    if st is None or len(st) == 0:
        return st
    keep = [tr for tr in st if not _station_is_excluded(tr.stats.station)]
    if len(keep) == len(st):
        return st
    out = obspy.Stream(keep)
    print(f"  Removed {len(st)-len(out)} traces from excluded/moving trigger stations")
    return out


## 5. Per-window processing functions

In [5]:
def run_detection_for_window(label, window_start, window_end, network, location, geometry, station_names, dirs, nominal_start=None, nominal_end=None):
    all_chunk_detection_files = []
    failed_chunks = []

    print("\n" + "#" * 88)
    print(f"DETECTION WINDOW: {label}")
    print(f"Deployment: {network}.{location}.{station}.{channel}")
    print(f"Window: {window_start} to {window_end} ({(window_end-window_start)/60:.1f} min)")
    print(f"Output: {dirs['outdir']}")
    memory_status("Before detection: ")

    for ichunk, (core_start, core_end) in enumerate(iter_time_chunks(window_start, window_end, chunk_seconds), start=1):
        read_start = max(window_start, core_start - overlap_seconds)
        read_end = min(window_end, core_end + overlap_seconds)

        print("\n" + "=" * 72)
        print(f"{label} chunk {ichunk}: core {core_start} to {core_end}; read {read_start} to {read_end}")
        memory_status("  ")

        chunk_tag = f"{ichunk:06d}_{core_start.strftime('%Y%m%dT%H%M%S')}_{core_end.strftime('%Y%m%dT%H%M%S')}"
        chunk_csv = dirs["chunks"] / f"{label}_{network}_{location}_{chunk_tag}_detections.csv"

        if chunk_csv.exists() and not reprocess_existing_chunks:
            print(f"  Already exists, skipping: {chunk_csv.name}")
            all_chunk_detection_files.append(chunk_csv)
            continue

        st_chunk = obspy.Stream()
        st_detect = obspy.Stream()
        try:
            st_chunk = read_deployment_from_sds(
                sds_root=sds_root,
                network=network,
                location=location,
                station=station,
                channel=channel,
                starttime=read_start,
                endtime=read_end,
                merge=True,
                verbose=False,
            )

            st_chunk = filter_excluded_stations(st_chunk)

            if len(st_chunk) == 0:
                print("  No traces read")
                pd.DataFrame().to_csv(chunk_csv, index=False)
                all_chunk_detection_files.append(chunk_csv)
                continue

            st_chunk = attach_station_geometry(st_chunk, geometry)
            st_detect = preprocess_for_detection(st_chunk, detect_cfg)
            df_chunk = detect_network_events(st_detect, detect_cfg)

            if df_chunk is None or len(df_chunk) == 0:
                df_chunk = pd.DataFrame()
                print("  No detections")
            else:
                df_chunk = filter_events_to_core_chunk(df_chunk, core_start, core_end)
                if len(df_chunk):
                    df_chunk["timewindow_label"] = label
                    df_chunk["network"] = network
                    df_chunk["location"] = location
                    df_chunk["chunk_index"] = ichunk
                    df_chunk["core_start"] = str(core_start)
                    df_chunk["core_end"] = str(core_end)
                    df_chunk["read_start"] = str(read_start)
                    df_chunk["read_end"] = str(read_end)
                    df_chunk["nominal_starttime"] = str(nominal_start) if nominal_start is not None else str(window_start)
                    df_chunk["nominal_endtime"] = str(nominal_end) if nominal_end is not None else str(window_end)
                    df_chunk["search_starttime"] = str(window_start)
                    df_chunk["search_endtime"] = str(window_end)
                    df_chunk["search_pad_before_s"] = SEARCH_PAD_BEFORE_S
                    df_chunk["search_pad_after_s"] = SEARCH_PAD_AFTER_S
                    if nominal_start is not None and nominal_end is not None:
                        df_chunk = add_padding_status(df_chunk, nominal_start, nominal_end)
                print(f"  Kept {len(df_chunk)} core detections")

            df_chunk.to_csv(chunk_csv, index=False)
            all_chunk_detection_files.append(chunk_csv)

        except Exception as exc:
            failed_chunks.append({
                "timewindow_label": label,
                "chunk_index": ichunk,
                "core_start": str(core_start),
                "core_end": str(core_end),
                "error": repr(exc),
            })
            err_file = dirs["logs"] / f"{label}_{network}_{location}_{chunk_tag}_ERROR.txt"
            err_file.write_text(traceback.format_exc())
            print(f"  ERROR in chunk {ichunk}: {exc}")
            print(f"  Wrote traceback: {err_file}")

        finally:
            try:
                del st_chunk, st_detect
            except Exception:
                pass
            gc.collect()

    failed_chunks_df = pd.DataFrame(failed_chunks)
    failed_chunks_csv = dirs["logs"] / f"{label}_{network}_{location}_failed_chunks.csv"
    failed_chunks_df.to_csv(failed_chunks_csv, index=False)

    print("\nDetection pass complete")
    print(f"Chunk detection files: {len(all_chunk_detection_files)}")
    print(f"Failed chunks: {len(failed_chunks_df)}")
    memory_status("After detection: ")
    return all_chunk_detection_files, failed_chunks_df


def combine_detections_for_window(label, network, location, dirs):
    chunk_frames = []
    pattern = f"{label}_{network}_{location}_*_detections.csv"
    for f in sorted(dirs["chunks"].glob(pattern)):
        try:
            df = pd.read_csv(f)
            if len(df):
                df["source_chunk_file"] = f.name
                chunk_frames.append(df)
        except pd.errors.EmptyDataError:
            pass

    df_events_raw = pd.concat(chunk_frames, ignore_index=True) if chunk_frames else pd.DataFrame()
    raw_events_csv = dirs["tables"] / f"{label}_{network}_{location}_detected_events_raw_chunked.csv"
    df_events_raw.to_csv(raw_events_csv, index=False)

    df_events = merge_duplicate_events(df_events_raw, duplicate_tolerance_s=duplicate_tolerance_s)
    if len(df_events):
        df_events["timewindow_label"] = label
        df_events["network"] = network
        df_events["location"] = location

        # Keep padding metadata if it exists in chunk CSVs. This is useful for
        # distinguishing events inside the original field-note interval from
        # detections found in the added clock/stack buffer.
        if "nominal_starttime" in df_events.columns and "nominal_endtime" in df_events.columns:
            try:
                df_events = add_padding_status(df_events, df_events["nominal_starttime"].iloc[0], df_events["nominal_endtime"].iloc[0])
            except Exception:
                pass

    events_csv = dirs["tables"] / f"{label}_{network}_{location}_detected_events_merged.csv"
    df_events.to_csv(events_csv, index=False)

    print(f"Raw detections: {len(df_events_raw)} -> merged detections: {len(df_events)}")
    print(f"Wrote {raw_events_csv}")
    print(f"Wrote {events_csv}")
    return df_events_raw, df_events, raw_events_csv, events_csv


In [6]:
def run_picking_for_window(label, window_start, window_end, network, location, geometry, station_names, dirs, df_events):
    all_method_pick_frames = []
    all_station_pick_frames = []
    failed_events = []

    if df_events is None or len(df_events) == 0:
        print(f"No events to pick for {label}")
        picks_df = pd.DataFrame()
        station_picks_df = pd.DataFrame()
        failed_events_df = pd.DataFrame()
    else:
        pick_read_pre_s = max(1.0, detect_cfg.pretrigger_seconds + pick_cfg.event_pre_pick_s + 0.5)
        pick_read_post_s = max(1.5, detect_cfg.posttrigger_seconds + pick_cfg.event_length_s + 0.75)
        start_col, end_col, center_col = event_time_columns(df_events)

        memory_status("Before picking: ")

        for i, row in df_events.iterrows():
            event_id = row.get("event_id", f"E{i+1:05d}")
            st_event = obspy.Stream()
            st_pick = obspy.Stream()
            try:
                event_start = to_utcdatetime(row[start_col])
                event_end = to_utcdatetime(row[end_col]) if end_col is not None else event_start
                read_start = max(window_start, event_start - pick_read_pre_s)
                read_end = min(window_end, event_end + pick_read_post_s)

                print(f"\nPicking {label} {event_id}: read {read_start} to {read_end}")
                st_event = read_deployment_from_sds(
                    sds_root=sds_root,
                    network=network,
                    location=location,
                    station=station,
                    channel=channel,
                    starttime=read_start,
                    endtime=read_end,
                    merge=True,
                    verbose=False,
                )

                st_event = filter_excluded_stations(st_event)

                if len(st_event) == 0:
                    print("  No traces")
                    continue

                st_event = attach_station_geometry(st_event, geometry)
                st_pick = preprocess_for_picking(st_event, pick_cfg)

                df_one_event = pd.DataFrame([row.to_dict()])
                picks_df_i, station_picks_df_i = run_station_picking(
                    st_pick=st_pick,
                    df_events=df_one_event,
                    station_names=station_names,
                    cfg=pick_cfg,
                    min_event_peak_amplitude=detect_cfg.min_event_peak_amplitude,
                    diagnostics_dir=dirs["figs"] / "station_pick_diagnostics",
                    make_diagnostic_plots=False,
                )

                if picks_df_i is not None and len(picks_df_i):
                    picks_df_i["timewindow_label"] = label
                    picks_df_i["network"] = network
                    picks_df_i["location"] = location
                    picks_df_i["event_id"] = event_id
                    all_method_pick_frames.append(picks_df_i)
                if station_picks_df_i is not None and len(station_picks_df_i):
                    station_picks_df_i["timewindow_label"] = label
                    station_picks_df_i["network"] = network
                    station_picks_df_i["location"] = location
                    station_picks_df_i["event_id"] = event_id
                    all_station_pick_frames.append(station_picks_df_i)

            except Exception as exc:
                failed_events.append({"timewindow_label": label, "event_id": event_id, "error": repr(exc)})
                err_file = dirs["logs"] / f"{label}_{network}_{location}_{event_id}_PICK_ERROR.txt"
                err_file.write_text(traceback.format_exc())
                print(f"  ERROR picking {event_id}: {exc}")

            finally:
                try:
                    del st_event, st_pick
                except Exception:
                    pass
                gc.collect()

        picks_df = pd.concat(all_method_pick_frames, ignore_index=True) if all_method_pick_frames else pd.DataFrame()
        station_picks_df = pd.concat(all_station_pick_frames, ignore_index=True) if all_station_pick_frames else pd.DataFrame()
        failed_events_df = pd.DataFrame(failed_events)

    picks_csv = dirs["tables"] / f"{label}_{network}_{location}_individual_method_picks.csv"
    station_picks_csv = dirs["tables"] / f"{label}_{network}_{location}_station_consensus_picks.csv"
    failed_events_csv = dirs["logs"] / f"{label}_{network}_{location}_failed_events.csv"

    picks_df.to_csv(picks_csv, index=False)
    station_picks_df.to_csv(station_picks_csv, index=False)
    failed_events_df.to_csv(failed_events_csv, index=False)

    print(f"\nWrote {picks_csv} ({len(picks_df)} rows)")
    print(f"Wrote {station_picks_csv} ({len(station_picks_df)} rows)")
    print(f"Wrote {failed_events_csv} ({len(failed_events_df)} rows)")
    memory_status("After picking: ")
    return picks_df, station_picks_df, failed_events_df


def save_event_mseed_for_window(label, window_start, window_end, network, location, geometry, dirs, station_picks_df):
    written_mseed_all = []
    written_figs_all = []
    failed_save_events = []

    if not write_event_mseed or station_picks_df is None or len(station_picks_df) == 0:
        print(f"No event MiniSEED to write for {label}")
        failed_save_events_df = pd.DataFrame()
    else:
        for event_id, station_picks_one in station_picks_df.groupby("event_id"):
            st_event = obspy.Stream()
            event_streams = []
            event_windows_df = pd.DataFrame()
            try:
                pick_time_col = infer_pick_time_column(station_picks_one)

                candidate_times = [to_utcdatetime(v) for v in station_picks_one[pick_time_col]]
                candidate_times = [t for t in candidate_times if t is not None]
                if not candidate_times:
                    raise ValueError("No valid pick times found")
                earliest_pick = min(candidate_times)

                read_start = max(window_start, earliest_pick - pick_cfg.event_pre_pick_s - 0.5)
                read_end = min(window_end, earliest_pick + pick_cfg.event_length_s + 0.5)

                print(f"Saving {label} {event_id}: read {read_start} to {read_end}")
                st_event = read_deployment_from_sds(
                    sds_root=sds_root,
                    network=network,
                    location=location,
                    station=station,
                    channel=channel,
                    starttime=read_start,
                    endtime=read_end,
                    merge=True,
                    verbose=False,
                )
                if len(st_event) == 0:
                    continue
                st_event = attach_station_geometry(st_event, geometry)

                event_streams, event_windows_df = trim_events_from_consensus_picks(
                    st_event,
                    station_picks_one,
                    pre_pick_s=pick_cfg.event_pre_pick_s,
                    event_length_s=pick_cfg.event_length_s,
                )

                event_windows_csv = dirs["tables"] / f"{label}_{network}_{location}_{event_id}_event_windows.csv"
                event_windows_df.to_csv(event_windows_csv, index=False)

                written_mseed = save_event_mseed_files(
                    event_streams,
                    dirs["events_mseed"],
                    prefix=f"{label}_{network}_{location}_{event_id}",
                )
                written_mseed_all.extend(written_mseed)

                if make_event_qc_figures:
                    written_figs = save_event_qc_figures(
                        event_streams,
                        station_picks_one,
                        dirs["figs"] / "event_station_consensus",
                        components=("Z", "N", "E"),
                        prefix=f"{label}_{network}_{location}_{event_id}",
                    )
                    written_figs_all.extend(written_figs)

            except Exception as exc:
                failed_save_events.append({"timewindow_label": label, "event_id": event_id, "error": repr(exc)})
                err_file = dirs["logs"] / f"{label}_{network}_{location}_{event_id}_SAVE_ERROR.txt"
                err_file.write_text(traceback.format_exc())
                print(f"  ERROR saving {event_id}: {exc}")

            finally:
                try:
                    del st_event, event_streams, event_windows_df
                except Exception:
                    pass
                gc.collect()

        failed_save_events_df = pd.DataFrame(failed_save_events)

    failed_save_events_csv = dirs["logs"] / f"{label}_{network}_{location}_failed_save_events.csv"
    failed_save_events_df.to_csv(failed_save_events_csv, index=False)

    print(f"Wrote {len(written_mseed_all)} event MiniSEED files")
    print(f"Wrote {len(written_figs_all)} QC figures")
    print(f"Failed save events: {len(failed_save_events_df)}")
    return written_mseed_all, written_figs_all, failed_save_events_df


## 6. Run the named-window batch

In [7]:
def run_one_timewindow(label, nominal_start, nominal_end):
    network, location = parse_label(label)
    dirs = make_output_dirs(label, network, location)
    geometry, station_names, geometry_sheet, stations_csv = load_geometry_for_deployment(network, location, dirs)

    search_start = nominal_start - SEARCH_PAD_BEFORE_S
    search_end = nominal_end + SEARCH_PAD_AFTER_S

    print(f"Loaded geometry sheet {geometry_sheet} for {len(station_names)} stations")
    print(f"Wrote station geometry: {stations_csv}")
    print(f"Nominal window: {nominal_start} to {nominal_end}")
    print(f"Search window:  {search_start} to {search_end} "
          f"(pad before={SEARCH_PAD_BEFORE_S}s, after={SEARCH_PAD_AFTER_S}s)")

    all_chunk_detection_files, failed_chunks_df = run_detection_for_window(
        label, search_start, search_end, network, location, geometry, station_names, dirs,
        nominal_start=nominal_start, nominal_end=nominal_end
    )
    df_events_raw, df_events, raw_events_csv, events_csv = combine_detections_for_window(
        label, network, location, dirs
    )
    picks_df, station_picks_df, failed_events_df = run_picking_for_window(
        label, search_start, search_end, network, location, geometry, station_names, dirs, df_events
    )
    written_mseed_all, written_figs_all, failed_save_events_df = save_event_mseed_for_window(
        label, search_start, search_end, network, location, geometry, dirs, station_picks_df
    )

    return {
        "label": label,
        "network": network,
        "location": location,
        "nominal_start": str(nominal_start),
        "nominal_end": str(nominal_end),
        "search_start": str(search_start),
        "search_end": str(search_end),
        "search_pad_before_s": SEARCH_PAD_BEFORE_S,
        "search_pad_after_s": SEARCH_PAD_AFTER_S,
        "duration_min_nominal": (nominal_end - nominal_start) / 60.0,
        "duration_min_search": (search_end - search_start) / 60.0,
        "outdir": str(dirs["outdir"]),
        "n_chunk_files": len(all_chunk_detection_files),
        "n_failed_chunks": len(failed_chunks_df),
        "n_raw_detections": len(df_events_raw),
        "n_merged_events": len(df_events),
        "n_method_picks": len(picks_df),
        "n_station_picks": len(station_picks_df),
        "n_failed_pick_events": len(failed_events_df),
        "n_event_mseed_files": len(written_mseed_all),
        "n_failed_save_events": len(failed_save_events_df),
        "events_csv": str(events_csv),
        "raw_events_csv": str(raw_events_csv),
    }

labels_to_run = list(timewindows.keys()) if RUN_LABELS is None else list(RUN_LABELS)
run_summaries = []

for label in labels_to_run:
    nominal_start, nominal_end = timewindows[label]
    summary = run_one_timewindow(label, nominal_start, nominal_end)
    run_summaries.append(summary)
    gc.collect()

summary_df = pd.DataFrame(run_summaries)
summary_csv = base_outdir / "timewindow_detection_run_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"\nWrote batch summary: {summary_csv}")
display(summary_df)


Loaded geometry sheet position_coded_sds_station_codes for 23 stations
Wrote station geometry: /Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows/T1_N1_Streamer/T1_N1_DPall/tables/T1_N1_position_coded_station_geometry.csv
Nominal window: 2026-05-16T17:13:00.000000Z to 2026-05-16T21:48:00.000000Z
Search window:  2026-05-16T17:12:00.000000Z to 2026-05-16T21:51:00.000000Z (pad before=60s, after=180s)

########################################################################################
DETECTION WINDOW: T1_N1_Streamer
Deployment: T1.N1.*.DP*
Window: 2026-05-16T17:12:00.000000Z to 2026-05-16T21:51:00.000000Z (279.0 min)
Output: /Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows/T1_N1_Streamer/T1_N1_DPall
Before detection: RAM used=9.1 GB, available=6.6 GB, percent=74.5%

T1_N1_Streamer chunk 1: core 2026-05-16T17:12:00.000000Z to 2026-05-16T17:13:00.000000Z; read 2026-05-16T17:12:00.000000Z to 2026-05-16T17:13:10.000000Z
  RAM used=9.1 GB, ava

,label,network,location,nominal_start,nominal_end,search_start,search_end,search_pad_before_s,search_pad_after_s,duration_min_nominal,...,n_failed_chunks,n_raw_detections,n_merged_events,n_method_picks,n_station_picks,n_failed_pick_events,n_event_mseed_files,n_failed_save_events,events_csv,raw_events_csv
0,T1_N1_Streamer,T1,N1,2026-05-16T17:13:00.000000Z,2026-05-16T21:48:00.000000Z,2026-05-16T17:12:00.000000Z,2026-05-16T21:51:00.000000Z,60,180,275.00,...,0,663,389,49320,8220,0,389,0,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...
1,T1_N2_Nodal1,T1,N2,2026-05-17T16:00:00.000000Z,2026-05-17T19:00:00.000000Z,2026-05-17T15:59:00.000000Z,2026-05-17T19:03:00.000000Z,60,180,180.00,...,0,1014,762,99588,16598,0,762,0,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...
2,T1_N2_Refraction1m,T1,N2,2026-05-18T16:03:54.000000Z,2026-05-18T18:38:33.000000Z,2026-05-18T16:02:54.000000Z,2026-05-18T18:41:33.000000Z,60,180,154.65,...,0,369,343,47334,7889,0,343,0,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...
3,T1_N2_Refraction2m,T1,N2,2026-05-18T20:18:44.000000Z,2026-05-18T23:12:53.000000Z,2026-05-18T20:17:44.000000Z,2026-05-18T23:15:53.000000Z,60,180,174.15,...,0,334,279,38502,6417,0,279,0,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...
4,T1_N2_Nodal2,T1,N2,2026-05-19T12:59:00.000000Z,2026-05-19T13:21:00.000000Z,2026-05-19T12:58:00.000000Z,2026-05-19T13:24:00.000000Z,60,180,22.00,...,0,57,57,7866,1311,0,57,0,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...
5,T1_N3_Nodal3,T1,N3,2026-05-19T13:59:00.000000Z,2026-05-19T14:15:00.000000Z,2026-05-19T13:58:00.000000Z,2026-05-19T14:18:00.000000Z,60,180,16.00,...,0,0,0,0,0,0,0,0,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...
6,T3_N4_Refraction1am,T3,N4,2026-05-19T16:02:00.000000Z,2026-05-19T18:25:00.000000Z,2026-05-19T16:01:00.000000Z,2026-05-19T18:28:00.000000Z,60,180,143.00,...,0,482,389,53682,8947,0,389,0,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...


## 7. Optional: combine all merged detections into one master CSV

In [8]:
master_frames = []
for label in labels_to_run:
    network, location = parse_label(label)
    dirs = make_output_dirs(label, network, location)
    events_csv = dirs["tables"] / f"{label}_{network}_{location}_detected_events_merged.csv"
    if events_csv.exists():
        df = pd.read_csv(events_csv)
        if len(df):
            master_frames.append(df)

master_events_df = pd.concat(master_frames, ignore_index=True) if master_frames else pd.DataFrame()
master_events_csv = base_outdir / "all_timewindows_detected_events_merged.csv"
master_events_df.to_csv(master_events_csv, index=False)
print(f"Wrote {master_events_csv} ({len(master_events_df)} events)")
display(master_events_df.head())


EmptyDataError: No columns to parse from file

## 8. Notes and tuning

- Start by setting `RUN_LABELS = ["T1_N2_Nodal2"]` for a short test window.
- Each label is resumable: existing chunk CSV files are skipped.
- If there are too many false detections, increase `threshold_on`, `min_snr`, or `min_channels`.
- If obvious shots are missed, lower `threshold_on` or `min_channels`, or test `channel = "DPZ"` first.
- If a crash or interruption occurs, rerun the notebook; completed chunk files should be reused.
